# Equity volatility calibration
How tQuant turns the listed option quotes of an underlying into an implied volatility surface and a repo curve, step
by step on the real code. The sections follow Models and Pricing, Market handles, *Equity volatilities* (§1.4).

Two underlyings, the two cases of the pipeline:
- **AAPL**: a single stock, American options, discrete dividends forecast by Bloomberg;
- **SX5E**: an index, European options, dividends implied by the options themselves (§4).

**Data.** The Bloomberg snapshot of 2026-11-13 of these two underlyings, frozen in `equity_vol_calibration_data.csv`
so that the notebook always shows the same numbers. Every row of the file is a row of one table of the snapshot,
the column `table` says which (`eq_option_quotes.csv`, `ir_curves.csv`, ...). The file also holds what the daily
calibration produced from these quotes (`eq_vol.csv`, `eq_repo.csv`, `eq_vol_calibration.csv`): along the way we
recompute each piece by hand and check that it is the same.

**The pipeline in production.** `mdbuild` (in `marketdata/`) downloads the quotes and applies the first data-quality
rules (§2); `tq.calibrate_snapshot_vols` then calls, per underlying, `tq.build_equity_vol`, which does §3, §5, §6 and
§7; for an index it first replaces the long-dated dividends (§4).

### Setup
Read the CSV, split it back into the snapshot tables (`frames`: file name → DataFrame) and build the market with
`MarketEnvironment.from_frames`, the same function that reads a snapshot folder. `OPTIONS` are the download rules of
`marketdata/config/settings.yaml` (which expiries and strikes are requested, when a quote is stale).

In [ ]:
import os
import sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "marketdata"))                   # mdbuild: the download rules

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import yaml
from scipy.stats import norm
import tensorquant as tq
from tensorquant.calibration import equityvol as ev             # internal steps of build_equity_vol (ev._...)
from mdbuild import options as mdo                              # download rules (expiries, strikes, stale quotes)

# one CSV -> the snapshot tables; columns that belong to other tables are all NaN and are dropped
DATA = pd.read_csv("equity_vol_calibration_data.csv", low_memory=False)
frames = {name: g.drop(columns="table").dropna(axis=1, how="all").reset_index(drop=True)
          for name, g in DATA.groupby("table")}
frames["eq_spot.csv"] = frames["eq_spot.csv"].reindex(columns=["ticker", "ccy", "spot", "repo", "div_yield"])
QUOTES = frames["eq_option_quotes.csv"]
EVALUATION_DATE = pd.Timestamp(QUOTES["quote_date"].iloc[0]).date()
mkt = tq.MarketEnvironment.from_frames(frames, EVALUATION_DATE)  # curves, spots, dividends of the snapshot
tq.Settings.evaluation_date = EVALUATION_DATE
OPTIONS = yaml.safe_load((REPO / "marketdata" / "config" / "settings.yaml").read_text())["options"]
# the data set was downloaded on 2026-11-13 with the targets of that day (now 30%-200% of F, chains 25%-250%)
OPTIONS.update(moneyness=[0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.975, 1.0, 1.025, 1.05, 1.1, 1.15, 1.2, 1.3, 1.4, 1.5],
               strike_range="40%-200%")
TICKER, INDEX = "AAPL_US", "SX5E"
print("tables:", {k: len(v) for k, v in frames.items()})
QUOTES.groupby("ticker").agg(options=("strike", "size"), expiries=("expiry", "nunique"),
                             exercise=("exercise", "first"), quote_date=("quote_date", "first"),
                             ref_spot=("ref_spot", "first"), spot_source=("spot_source", "first"))

**Reading the result.** 323 AAPL options on 12 expiries and 395 SX5E options on 13 expiries, all quoted on
2026-11-13. `ref_spot` is the spot read in the same Bloomberg request as the options (`with the quotes`): the
snapshot spot is read minutes earlier, and for put-call parity spot and option prices must be of the same moment.

## 1. Construction from listed options
The calibration works **one expiry at a time**, in three stages:
1. the **forward** of the expiry, implied from calls and puts by put-call parity (§3); with the dividends fixed, what
   is left of the carry is the **repo**;
2. the **smile** of the expiry: a lognormal-mixture density fitted to the out-of-the-money quotes (§5, §6);
3. the **surface**: the smiles of all the expiries, interpolated and extrapolated onto a fixed grid (§7).

The inputs are the quotes and an `Equity` *view* of the underlying, built as `calibrate_snapshot_vols` builds it:
- the spot of the quotes (`ref_spot`);
- the discount curve of the currency (SOFR for AAPL);
- the dividends forecast by Bloomberg;
- **no repo**: the repo is what the options will tell us (§3).

In [ ]:
def equity_inputs(ticker):
    """Quotes of *ticker* and its view as in calibrate_snapshot_vols: spot of the quotes, repo left to the options."""
    q = QUOTES[QUOTES["ticker"] == ticker].copy()
    q["expiry"] = pd.to_datetime(q["expiry"]).dt.date
    e = mkt.get_equity(ticker)                                      # market view: discount curve, dividends
    spot = tf.Variable(float(q["ref_spot"].iloc[0]), dtype=tf.float64)
    return q, tq.Equity(ticker, e.currency, spot, e.discount_curve, None, e.dividends, e.div_yield, None)


quotes, view = equity_inputs(TICKER)
eq = mkt.get_equity(TICKER)
QUOTE_DATE = pd.Timestamp(quotes["quote_date"].iloc[0]).date()
REF_SPOT = float(view.spot)
print(f"{TICKER}: {len(quotes)} quotes, spot {REF_SPOT} ({quotes['spot_source'].iloc[0]}), "
      f"{len(eq.dividends.ex_dates) if eq.dividends else 0} projected dividends")
quotes.head(8)

**Reading the result.** One row per option: expiry, strike, type, exercise style, `bid` / `ask` (the tradable
quotes), `last` (the last price, used as a *settlement* when there is no bid / ask, §2.3), `last_update` (the date of
the last bid update) and the Bloomberg ticker.

## 2. The quotes
Before any model, which quotes we use and how reliable each one is. A full chain has thousands of options (AAPL lists
dozens of expiries and hundreds of strikes). Only a selection is requested: twenty to thirty options per expiry, out of
the chain of the options listed with strikes between 40% and 200% of the spot (the settings of the download of this
data set; the downloads now request 25%-250% and targets from 30% to 200% of the forward) (on an expiry listed both in the standard
series and in a weekly one, the standard one).

Keeping or discarding a quote happens in **two stages**:
- **at the download** (`mdbuild`): the selection of expiries and strikes (§2.1, §2.2) and the stale quotes (§2.3).
  The CSV holds the result: these sections do not filter anything, they show the rules and check that they give
  back the quotes we have;
- **at the calibration** (`build_equity_vol`): which of the downloaded quotes are used and with which weight (§2.4).
  This is the first filter applied in this notebook.

### 2.1 Expiries
The rule of `mdbuild.options.select_expiries`, with the parameters of `settings.yaml`:
- the first `weekly_expiries` listed expiries, weeklies included: the very short end;
- the standard monthlies (third Friday) up to `monthly_months`;
- the quarterly monthlies (March, June, September, December) up to `quarterly_months`: the most liquid ones;
- then one expiry a year, the nearest to mid-December, up to `max_years`.

The cell labels each expiry with the rule that selected it, then applies `select_expiries` to them again to check
that the set is the same.

In [ ]:
print({k: OPTIONS[k] for k in ("weekly_expiries", "monthly_months", "quarterly_months", "max_years")})
expiries = sorted(quotes["expiry"].unique())
first = expiries[:OPTIONS["weekly_expiries"]]
monthly_end = mdo._add_months(QUOTE_DATE, OPTIONS["monthly_months"])
quarterly_end = mdo._add_months(QUOTE_DATE, OPTIONS["quarterly_months"])


def rule(e):
    """The rule of select_expiries that keeps expiry e (checked in this order)."""
    if e in first:
        return "first listed"
    if mdo._monthly(e) and e <= monthly_end:                       # third Friday
        return "monthly"
    if mdo._monthly(e) and e.month in (3, 6, 9, 12) and e <= quarterly_end:
        return "quarterly"
    return "yearly (nearest to 15 Dec)"


reselected = mdo.select_expiries(expiries, QUOTE_DATE, OPTIONS["weekly_expiries"], OPTIONS["monthly_months"],
                                 OPTIONS["quarterly_months"], OPTIONS["max_years"])
print("select_expiries keeps every downloaded expiry:", reselected == expiries)
pd.DataFrame({"expiry": expiries, "days": [(e - QUOTE_DATE).days for e in expiries], "rule": [rule(e) for e in expiries]})

**Reading the result.**
- The first two expiries are at 3 and 5 days.
- The monthlies run to January 2027, then the quarterlies to March 2028.
- Beyond that, AAPL lists only December 2028 and January 2029, so the last calibrated expiry is at about
  2.2 years.

The longer maturities of the surface will therefore be extrapolated (§7). SX5E, by contrast, lists to 2032.

### 2.2 Strikes
The strikes are chosen around an estimate of the forward,
$$\hat F = (S_0 - D)\,e^{rT},$$
with $D$ the sum of the forecast dividends with ex-date up to $T$ (not discounted) and $r$ the overnight fixing of the
currency (SOFR for USD). $\hat F$ is only a guide for the selection: the forward used afterwards is implied by the
options themselves (§3). **Moneyness here is $K/\hat F$.** For each expiry:
- **Smile options.** For each target moneyness $m$ in `moneyness` (50, 60, 70, 80, 85, 90, 95, 97.5, 100, 102.5, 105,
  110, 115, 120, 130, 140 and 150%), the target strike is $m\hat F$. Among the listed **out-of-the-money** options
  (puts with $K < \hat F$ for $m < 100\%$, calls with $K \ge \hat F$ for $m \ge 100\%$), the one whose strike is the
  nearest to it is taken. For example, with $\hat F = 360$ and strikes every 5, the 80% target 288 gives the 290 put.
  Where the strikes are sparse (wings, long expiries):
  - the listed strike can be some way from the target;
  - two targets can give the same option (taken once);
  - a target beyond the listed strikes gives the outermost one.

  These options carry the smile, usually one option type per strike.
- **Parity options.** Around $\hat F$, the `parity_strikes` = 4 listed strikes below it and the 4 at or above it
  (8 strikes, among those listed with both a call and a put), and on each of them **both the call and the put**:
  16 options, for the put-call parity of the forward (§3). Near the forward both options are liquid and their
  spreads narrow.
- **In all**, per expiry: at most 17 smile options and the 16 parity options, an option counted once when it is in
  both (the smile options from about 95% to 105% of $\hat F$ are parity options too): typically 20 to 30 options.

The cell recomputes $\hat F$ with `mdbuild.options.forward_estimate` (`marketdata/mdbuild/options.py`). The CSV does
not say which rule selected each option, so the label is rebuilt: a strike quoted with both a call and a put is
taken as a parity strike.

In [ ]:
print("moneyness targets:", OPTIONS["moneyness"], "| parity strikes:", OPTIONS["parity_strikes"])
fixings = frames["ir_fixings.csv"]
rate = float(fixings.loc[fixings["ccy"] == quotes["ccy"].iloc[0], "value"].iloc[0])      # overnight fixing (SOFR)
divs = frames["eq_dividends.csv"].query("ticker == @TICKER")
f_hat = {e: mdo.forward_estimate(REF_SPOT, divs, rate, QUOTE_DATE, e) for e in expiries}
quotes["m_hat"] = quotes["strike"] / quotes["expiry"].map(f_hat)                         # moneyness K / F_hat
both = quotes.groupby(["expiry", "strike"])["option_type"].transform("nunique") == 2   # call and put on the strike
quotes["role"] = np.where(both, "parity (C and P)", "smile (OTM)")
summary = quotes.groupby("expiry").agg(calls=("option_type", lambda s: (s == "C").sum()),
                                       puts=("option_type", lambda s: (s == "P").sum()),
                                       parity_strikes=("role", lambda s: (s == "parity (C and P)").sum() // 2),
                                       m_min=("m_hat", "min"), m_max=("m_hat", "max"))
summary.insert(0, "F_hat", summary.index.map(f_hat))
summary.round(3)

**Reading the result.** 23 to 30 options an expiry, 8 parity strikes (4 on each side of $\hat F$), moneyness
from 50% to 150%. The two weeklies are narrower (75%-117%): at a few days the far strikes are not listed or not
quoted. The plot below shows the same thing per expiry: the circles are the smile options, the squares the
parity strikes, tight around $K/\hat F = 1$.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = {e: i for i, e in enumerate(expiries)}
for (cp, role), g in quotes.groupby(["option_type", "role"]):
    ax.scatter(g["expiry"].map(x) + (0.12 if cp == "C" else -0.12), g["m_hat"], s=14,
               marker="o" if role.startswith("smile") else "s", label=f"{cp} {role}")
ax.axhline(1, color="grey", lw=0.8)
ax.set_xticks(range(len(expiries)), [str(e) for e in expiries], rotation=60)
ax.set_ylabel("K / F_hat"); ax.set_title(f"{TICKER}: the requested strikes"); ax.legend(fontsize=8); plt.show()

### 2.3 Quote date, spot and stale quotes (download)
For each option Bloomberg returns the bid, the ask, the last price and the time of the last bid update
(`LAST_UPDATE_BID_RT`: a time means today, a date an earlier session). The spot is read in the same request, so that
spot and options are of the same moment. Then, in `mdbuild.options.option_quotes`:
1. **Quote date.** The most frequent date of the last bid update among the options with a bid and an ask. If it is
   not today (market not open yet), the spot is the close of that day, so that spot and options belong to the same
   session.
2. **Stale quotes.** A bid / ask whose last update is missing or older than `stale_days` = 3 calendar days before the
   quote date is removed: a price that has not moved for days is not a market.
3. **Kept or dropped.** An option is kept if it still has both a bid and an ask, or a positive last price;
   otherwise it is dropped.

So an option in the CSV is one of three cases:
- a full bid / ask;
- only one side (kept here because it has a last price, discarded later in §2.4);
- only the last price (a future **settlement**, §2.4).

The cell counts them per expiry, then shows how old the updates are.

In [ ]:
print("stale_days:", OPTIONS["stale_days"], "| quote date", QUOTE_DATE, "| spot", REF_SPOT, quotes["spot_source"].iloc[0])
quotes["kind"] = np.select([quotes["bid"].notna() & quotes["ask"].notna(), quotes["bid"].notna() | quotes["ask"].notna(),
                            quotes["last"] > 0], ["bid / ask", "one-sided", "last only (settlement)"], "nothing")
cols = ["last_update"] + (["last_update_time"] if "last_update_time" in quotes else [])
display(pd.crosstab(quotes["expiry"], quotes["kind"]))
quotes.groupby("kind")[cols].agg(lambda s: s.value_counts().head(3).to_dict())

**Reading the result.**
- **AAPL.** Every quote was updated on the quote date, and no option fell back to a settlement price. The 25
  one-sided quotes sit on the short expiries: far out-of-the-money options with a bid of zero, so no bid at all.
- **Less liquid underlyings.** On European single stocks (for example ENI, in the live snapshots) and on the long
  SX5E expiries whole expiries have only settlement prices. §2.4 drops them when the underlying has a bid / ask
  market.

### 2.4 Which quotes enter the calibration
The first filter of the calibration, at the start of `build_equity_vol` (`ev._prepare_quotes`). Each downloaded
option is one of:

| case | condition | used as | weight |
|---|---|---|---|
| two-sided | bid $> 0$ and bid $\le$ ask $\le 3\times$ bid | mid $=(\text{bid}+\text{ask})/2$ | 1 |
| settlement | neither a bid nor an ask (removed as stale, or never quoted), last $> 0$ | discarded, unless the underlying has no bid / ask market (then bid = ask = last) | |
| one-sided | bid or ask only | discarded | |
| zero bid | bid $= 0$ | discarded | |
| crossed | ask $<$ bid | discarded (bad quote) | |
| too wide | ask $> 3\times$ bid | discarded (the mid is uncertain by more than half of itself) | |

A quote with a bid or an ask is **never replaced by its last price**: a too wide or one-sided quote is discarded
even when it has a last.

**Settlement prices** (Bloomberg's rule): an exchange mark, not a market, and it may be stale against the spot of
the quotes. They are used only for an underlying **without a bid / ask market**: an expiry has a good market with at
least 3 two-sided quotes and 10% of its quotes; with at least 3 such expiries (or 10% of them) the settlement prices
are dropped. Forwards, repo, smiles and the index dividends then come from the quoted expiries only; beyond the last
one the repo stays flat and the index dividends are projected (§4).

Who uses what, afterwards:
- **the forward** (§3): the two-sided quotes of the parity options;
- **the smile** (§6): the out-of-the-money options, with further checks on their implied vols.

The cell applies the rules of the table itself and compares the count with what `_prepare_quotes` keeps.

In [ ]:
bid, ask, last = quotes["bid"], quotes["ask"], quotes["last"]
quotes["status"] = np.select(                                       # the cases of the table, in order
    [(bid > 0) & (ask >= bid) & (ask <= 3 * bid), bid.isna() & ask.isna() & (last > 0), bid.isna() | ask.isna(),
     bid == 0, ask < bid],
    ["two-sided", "settlement", "one-sided", "zero bid", "crossed"], "too wide: ask > 3 bid")
prepared = ev._prepare_quotes(quotes)                               # the filter of build_equity_vol
print("two-sided here:", (quotes["status"] == "two-sided").sum(), "| kept by _prepare_quotes:", len(prepared),
      "| settlement prices kept:", int(prepared["settlement"].sum()))
pd.crosstab(quotes["expiry"], quotes["status"], margins=True)

**Reading the result.** 290 of the 323 AAPL quotes are kept, all of them two-sided. The 33 discarded ones
(25 one-sided, 8 too wide) are cheap wing options on the short and medium expiries, where the spread is as large
as the price.

## 3. Forward and repo of an expiry
**Why the forward comes from the options.** The forward $F(T) = \mathbb E[S_T]$ depends on rates, dividends and the
repo (the cost of borrowing the stock). The repo is not observable, and the forecast dividends can be wrong. The
options, though, price the forward directly. By put-call parity on European options at the same strike,
$$C - P = P(0,T)\,(F - K) \quad\Rightarrow\quad F = K + \frac{C - P}{P(0,T)}.$$

**The steps of `tq.implied_forward`:**
1. Start from the forward of the view.
2. Take the two strikes with a two-sided call *and* put that bracket it.
3. Compute the parity forward at each strike from the mid prices, and take their average.
4. Repeat (secant steps) until the forward lies between its two strikes and stops moving.

**American options.** Parity holds only for European options: the American mids are first **de-Americanized**,
turned into European prices (explained in §3.1, after the forward of the example).

**Bid / ask.** The bid / ask prices give a range for the forward:
- lowest: call bid and put ask;
- highest: call ask and put bid.

**The repo.** With the forward fixed and the forecast dividends kept, the rest of the carry is the repo:
$$q_r(T) = -\frac1T\ln\frac{F(T)\,P(0,T)}{S_0 - \mathrm{PV_{div}}(T)}.$$

### 3.1 One expiry by hand
The expiry nearest to one year. First `implied_forward` as the calibration calls it, using the two-sided quotes
only (AAPL has a bid / ask market: no settlement price is used, §2.4).

In [ ]:
EXPIRY = min(expiries, key=lambda e: abs((e - QUOTE_DATE).days - 365))   # about one year
grp = prepared[prepared["expiry"] == EXPIRY]                             # filtered quotes of the expiry
market = grp[~grp["settlement"]]                                          # two-sided ones only
res = tq.implied_forward(view, EXPIRY, market)                     # dividend_model "continuous", as build_equity_vol
t = view.year_fraction(EXPIRY)
df = float(view.discount_curve.discount(EXPIRY))                   # P(0, T) on SOFR
r = -np.log(df) / t
pv_div = float(view.pv_dividends(EXPIRY))                          # PV of the forecast dividends to T
F = res["forward"]
print(f"{EXPIRY}: T = {t:.4f}, P(0,T) = {df:.6f}, PV div = {pv_div:.4f}")
{k: res[k] for k in ("forward", "repo", "forward_bid", "forward_ask", "strike_low", "strike_high", "iterations", "converged")}

**Reading the result.**
- **Expiry:** December 2027 ($T \approx 1.09$).
- **Forward:** 376.58, between the strikes 375 and 380. Its bid / ask range runs from 375.28 to 377.96, about
  ±0.35%.
- **Repo:** −0.27%.

#### De-Americanizing a price
An American option can be exercised at any time up to the expiry, so it is worth at least the European one: its price
is the European price plus an **early-exercise premium** $\pi$,
$$V^{\text{Am}}(K) = V^{\text{Eu}}(K) + \pi(K), \qquad \pi(K) \ge 0.$$
Put-call parity holds only for European prices: on American mids it would put the premia into the forward. The premium
is not quoted anywhere, so it is computed with a model: Black-Scholes, with the Bjerksund-Stensland (1993) closed-form
approximation for the American exercise.

**What is known and what is not.** Known from the start: the spot $S_0$ (read with the quotes), strike $K$ and
expiry $T$, the rate $r = -\ln P(0,T)/T$ from the discount curve, and the market mids. Unknown: the vol $\sigma$ and
the forward $F$, which is what we are looking for and which also sets the model's carry $q$. The two unknowns are
handled differently:
- **the vol is not needed in advance**: with $F$ fixed, each option gives one equation (model price = mid) in one
  unknown, $\sigma$. It is an *output* of the de-Americanization, a vol per option used only to compute its premium
  (the smile vols of §6 are computed later, on the final forward);
- **the forward is guessed and then iterated**: start from the forward of the forecast dividends with a zero repo,
  de-Americanize on it, get a new forward from the parity, de-Americanize again on that one, and so on.

**One step, on the current forward $F$:**
1. **Inputs.** $S_0$, $r$, and the continuous carry $q = r - \ln(F/S_0)/T$, chosen so that the model forward
   $S_0\,e^{(r-q)T}$ is $F$. Dividends and repo enter only through $F$, the dividends as a yield (the default
   *continuous* dividend model; `dividend_model="discrete"` uses the spot net of the dividends instead).
2. **American implied vol** $\sigma_A$ of each of the four mids (call and put on the two strikes around $F$): the
   vol at which the American model price equals the market mid,
   $V^{\text{Am}}_{\text{model}}(\sigma_A) = \text{mid}$ (`tq.implied_volatility(..., american=True)`: prices on a
   grid of vols from 2% to 300% bracket the root, then regula falsi steps to $10^{-12}$).
3. **European price** at the same vol and inputs, $V^{\text{Eu}}_{\text{BS}}(\sigma_A)$.
4. **Premium** $\pi = \text{mid} - V^{\text{Eu}}_{\text{BS}}(\sigma_A)$, and the European mid is $\text{mid} - \pi$.
   The same premium is taken off the bid and the ask, for the bid / ask range of the forward.
5. **Parity** on the European mids: a new forward at each strike, their mean is the new $F$.

Steps 1-5 are repeated on the new $F$ until it stops moving (and lies between its two strikes). It converges in a
few steps because the premium depends little on $F$: a forward wrong by 0.3% moves the premium by a few cents, so
each new forward is much closer than the previous one. `implied_forward` uses secant steps on top of this plain
repetition, which only makes it faster.

**Why the result is reliable.** The model explains only the premium; the vol that drives it is the market's, at that
strike. A different American model would change the premium by a small fraction of itself, and the premium is
itself a few percent of the price at most.

**When the premium matters.**
- **Puts.** Exercising a put early gives the strike today, and the interest on it. That pays for deep in-the-money
  puts when rates are high: with USD rates around 4.5%, AAPL puts have a large premium.
- **Calls.** Exercising a call early gives the stock, so it pays only to receive a dividend. Without carry
  ($q \le 0$) a call is never exercised early and is priced as European. In the continuous dividend model the
  dividends are spread as a yield, so the premium just before a large single dividend is only approximate.
- **Index options** (SX5E, SPX) are European: there is nothing to remove, and the step is skipped.
- **Failure.** If a mid is outside the model range (below its price at 2% vol, for example a price below the
  intrinsic value, or above the one at 300%), its vol cannot be computed. The expiry then gets no forward
  ("American mid prices out of the model range"; it happens on the longest expiries of some illiquid names).

The iteration on the 1Y expiry, by hand with steps 1-5 (plain repetition, no secant), from the forward of the
forecast dividends with a zero repo:

In [ ]:
from tensorquant.markethandles.utils import OptionType

calls_q = market[market["option_type"] == "C"].set_index("strike")
puts_q = market[market["option_type"] == "P"].set_index("strike")
both_k = np.array(sorted(set(calls_q.index) & set(puts_q.index)))   # strikes with a two-sided call and put
F_it = float(view.forward(EXPIRY))                                   # start: forecast dividends, repo 0
steps = []
for it in range(6):
    pair_it = np.array([both_k[both_k <= F_it][-1], both_k[both_k > F_it][0]])   # the two strikes around F
    q_it = r - np.log(F_it / REF_SPOT) / t                                         # 1. carry on the current F
    euro, info = {}, {}
    for cp, tab, otype in (("C", calls_q, OptionType.Call), ("P", puts_q, OptionType.Put)):
        mid = 0.5 * (tab.loc[pair_it, "bid"] + tab.loc[pair_it, "ask"]).to_numpy()
        vol_it = tq.implied_volatility(mid, pair_it, cp, REF_SPOT, r, q_it, t, american=True)   # 2.
        euro[cp] = ev._prices(vol_it, pair_it, otype, REF_SPOT, r, q_it, t, False)              # 3.
        info[cp] = (vol_it, mid - euro[cp])                                                     # 4. premia
    F_new = float((pair_it + (euro["C"] - euro["P"]) / df).mean())                              # 5. parity
    steps.append({"step": it, "F used": F_it, "strikes": tuple(pair_it),
                  "put vols": tuple(np.round(info["P"][0], 4)), "put premia": tuple(np.round(info["P"][1], 4)),
                  "call premia": tuple(np.round(info["C"][1], 4)), "new F": F_new, "change": F_new - F_it})
    F_it = F_new
print(f"implied_forward: {F:.4f}")
pd.DataFrame(steps).round(4)

**Reading the result.**
- **Step 0.** The starting forward is about 1.1 dollars (0.3%) too low, because it has no repo. Even so, the put
  premia computed on it (1.95 and 2.19) are already close to the final ones (2.06 and 2.31).
- **Step 1.** The forward is within 0.12 of the solution.
- **Step 3.** It is stable to the cent, equal to `implied_forward`.

The vols are found along the way at each step: they were never needed as inputs.

The plots for the 1Y expiry:
- **Left:** the put on the higher strike of the pair, priced by the American and the European model as a function
  of the vol, on the forward found above. The market mid fixes $\sigma_A$ on the American curve; the European curve
  at $\sigma_A$ gives the European price; the gap is the premium.
- **Right:** the premium of every two-sided option of the expiry, in percent of its price, against $K/F$.

In [ ]:
q_carry = r - np.log(F / REF_SPOT) / t                              # carry on the final forward F
K_put = res["strike_high"]
put = market[(market["option_type"] == "P") & (market["strike"] == K_put)].iloc[0]
put_mid = 0.5 * (put["bid"] + put["ask"])
sig = np.linspace(0.05, 0.6, 200)
am = ev._prices(sig, np.full_like(sig, K_put), OptionType.Put, REF_SPOT, r, q_carry, t, True)    # American model
eu = ev._prices(sig, np.full_like(sig, K_put), OptionType.Put, REF_SPOT, r, q_carry, t, False)   # European model
sigma_a = float(tq.implied_volatility([put_mid], [K_put], "P", REF_SPOT, r, q_carry, t, american=True)[0])   # 2.
eu_a = float(ev._prices([sigma_a], [K_put], OptionType.Put, REF_SPOT, r, q_carry, t, False)[0])              # 3.

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(sig, am, label="American model (Bjerksund-Stensland)")
ax[0].plot(sig, eu, label="European model (Black-Scholes)")
ax[0].axhline(put_mid, color="grey", lw=0.8, ls=":", label=f"market mid {put_mid:.3f}")
ax[0].plot([sigma_a], [put_mid], "ko")
ax[0].plot([sigma_a], [eu_a], "ks")
ax[0].vlines(sigma_a, eu_a, put_mid, color="r", lw=2, label=f"premium {put_mid - eu_a:.3f}")   # 4.
ax[0].set_xlim(sigma_a - 0.08, sigma_a + 0.08)
ax[0].set_ylim(eu_a - 6, put_mid + 6)
ax[0].set_xlabel("vol"); ax[0].set_ylabel("price"); ax[0].legend(fontsize=8)
ax[0].set_title(f"{TICKER} {EXPIRY} put {K_put:g}: sigma_A = {sigma_a:.2%}")
for cp in ("P", "C"):
    side = market[market["option_type"] == cp].sort_values("strike")
    k, mids = side["strike"].to_numpy(float), 0.5 * (side["bid"] + side["ask"]).to_numpy(float)
    vols_a = tq.implied_volatility(mids, k, cp, REF_SPOT, r, q_carry, t, american=True)
    otype = OptionType.Put if cp == "P" else OptionType.Call
    eu_k = ev._prices(vols_a, k, otype, REF_SPOT, r, q_carry, t, False)
    ax[1].plot(k / F, 100 * (mids - eu_k) / mids, "o-", ms=4, label="puts" if cp == "P" else "calls")
ax[1].axvline(1, color="grey", lw=0.8)
ax[1].set_xlabel("K / F"); ax[1].set_ylabel("premium, % of the price"); ax[1].legend(fontsize=8)
ax[1].set_title("Early-exercise premium across the strikes")
plt.tight_layout(); plt.show()

**Reading the result.**
- **Left.** The market mid of the 380 put gives $\sigma_A = 26.6\%$, where the European price is 2.3 lower: that
  is the premium. The gap between the two curves shrinks as the vol grows, from 5.8 at 10% vol to 2.0 at 60%. At low
  vol the put is close to its intrinsic value and early exercise is almost certain to pay; at high vol the option is
  worth more alive.
- **Right, puts.** The premium grows from 0.5% of the price at 49% of the forward to 5.8% at 104%: the deeper in
  the money, the more likely the exercise.
- **Right, calls.** The premium is zero: AAPL's carry is almost nil ($q \approx 0.01\%$), so receiving the
  dividends never pays for giving up the time value.

Last, the final step on the final forward, in detail: steps 2-5 on the two calls and the two puts, then the repo
from the formula above.

In [ ]:
def bs(S, K, t, r, q, vol, cp):
    """European Black-Scholes price with continuous carry q."""
    d1 = (np.log(S / K) + (r - q + 0.5 * vol**2) * t) / (vol * np.sqrt(t))
    d2 = d1 - vol * np.sqrt(t)
    call = S * np.exp(-q * t) * norm.cdf(d1) - K * np.exp(-r * t) * norm.cdf(d2)
    return call if cp == "C" else call - S * np.exp(-q * t) + K * np.exp(-r * t)


pair = np.array([res["strike_low"], res["strike_high"]])
rows = {}
for cp in ("C", "P"):
    side = market[market["option_type"] == cp].set_index("strike").loc[pair]
    mid = 0.5 * (side["bid"] + side["ask"]).to_numpy()
    vol = tq.implied_volatility(mid, pair, cp, REF_SPOT, r, q_carry, t, american=True)   # 1. American vol
    european = bs(REF_SPOT, pair, t, r, q_carry, vol, cp)                                # 2. European price
    rows[cp] = european
    print(cp, "mid", mid.round(4), "| American vol", vol.round(4), "| European", european.round(4),
          "| early exercise premium", (mid - european).round(4))                         # 3. premium
parity = pair + (rows["C"] - rows["P"]) / df                        # F = K + (C - P) / P(0, T) at each strike
print("parity forwards", parity.round(5), "-> mean", parity.mean().round(5), "| implied_forward", round(F, 5))
print(f"repo = -ln(F P / (S - PV div)) / T = {-np.log(F * df / (REF_SPOT - pv_div)) / t:.5%} (implied_forward {res['repo']:.5%})")

**Reading the result.**
- **Calls:** no early-exercise premium. With a small dividend it never pays to exercise an AAPL call early.
- **Puts:** a premium of about 2 dollars (5% of the price). With rates around 4.5%, exercising a deep put early
  earns interest on the strike.

Ignoring the put premium would move the forward by about 2 dollars, about 0.5%. As a repo, that is an error of
about 50 bp.

The two parity forwards (376.54 and 376.62) differ by less than the bid / ask range. Their mean is exactly the
forward of `implied_forward`, and the repo matches too.

### 3.2 Every expiry: forwards, bid / ask range, repo
The same on every expiry. The repo becomes a curve:
- **Pillars:** one per expiry from 30 days on.
- **Interpolation:** linear in the zero repo rate.
- **Expiries under 30 days:** they keep their own forward but give no repo pillar. Over a few days a 0.05% error
  in the forward is already a 5% rate.

In [ ]:
fwd_rows = []
for e, g in prepared.groupby("expiry"):
    m = g[~g["settlement"]]
    try:
        o = tq.implied_forward(view, e, m if len(m) else g)               # settlement prices only as a fallback
    except ValueError as exc:
        fwd_rows.append({"expiry": e, "error": str(exc)[:60]}); continue
    fwd_rows.append({"expiry": e, "days": (e - QUOTE_DATE).days, "source": "market" if len(m) else "settlement",
                     "F_hat": f_hat[e], "forward": o["forward"],
                     "range bp": 1e4 * (o["forward_ask"] - o["forward_bid"]) / o["forward"],
                     "repo %": 100 * o["repo"], "iter": o["iterations"], "converged": o["converged"]})
forwards = pd.DataFrame(fwd_rows)
forwards.round(4)

**Reading the result.**
- **`F_hat` vs `forward`.** They differ by the repo and by the discounting of the dividends: the selection guess
  was close enough to pick the strikes.
- **`range bp`.** The width of the forward's bid / ask range. It grows with the expiry: the spreads of
  long-dated options are wider in price.

The plots below show the repo per expiry next to the pillars stored in `eq_repo.csv` (the crosses fall on the
pillars), and the width of the range.

In [ ]:
stored_repo = frames["eq_repo.csv"].query("ticker == @TICKER")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ok = forwards["days"] >= 30
ax[0].plot(forwards["days"], forwards["repo %"], "o:", label="every expiry")
ax[0].plot(forwards.loc[ok, "days"], forwards.loc[ok, "repo %"], "o-", label="pillars (>= 30 days)")
ax[0].plot([(pd.Timestamp(p).date() - QUOTE_DATE).days for p in stored_repo["pillar"]], 100 * stored_repo["rate"], "x",
           color="k", label="eq_repo.csv")
ax[0].set_xlabel("days"); ax[0].set_ylabel("repo %"); ax[0].legend(fontsize=8); ax[0].set_title("Repo per expiry")
ax[1].bar(forwards["days"].astype(str), forwards["range bp"]); ax[1].set_title("Bid / ask range of the forward (bp of F)")
ax[1].tick_params(axis="x", rotation=60); plt.tight_layout(); plt.show()

## 4. Index dividends: implied by the options, the repo held
For a stock, the forecast dividends are kept and the repo absorbs the rest of the carry (§3). For an index, that
fails at long maturities:
- Bloomberg's dividend projections for SX5E cover only 2-3 years.
- Beyond them, the forward of the options already contains the dividends that the projections lack.
- The repo would absorb them and fall to unrealistic levels (below −1.5% in 2032 on the settlement prices of the
  long expiries).

Market practice is the opposite: **the dividends come from the market, the repo is a stable financing level.**

**The steps of `implied_index_dividends`:**
1. Keep the projections up to `DIVIDEND_KEEP_YEARS` (2 years, where they are reliable).
2. Hold the repo at $q_0$, the repo implied at the last listed expiry inside those 2 years.
3. Beyond, each listed expiry gives the present value of the dividends up to it:
   $$\mathrm{PV_{div}}(T) = S_0 - F(T)\,P(0,T)\,e^{q_0T}.$$
4. Spread the increase between two expiries over the dividend dates of the period. The pattern is that of the
   projections, their last year repeated (the May peak of the European dividends).
5. After the last expiry, extrapolate the dividends to 10 years at a constant yield.

The cell starts from the raw Bloomberg download (`eq_dividends_download.csv`), redoes the whole step and compares
the result with the dividends of the snapshot.

In [ ]:
iq = QUOTES[QUOTES["ticker"] == INDEX].copy()
download = frames["eq_dividends_download.csv"].query("ticker == @INDEX").sort_values("ex_date")
ieq = mkt.get_equity(INDEX)
idx_div = tq.DividendCurve.from_dataframe(reference_date=mkt.evaluation_date, df=download, ex_date_col="ex_date",
                                          amount_col="amount", declared_date_col=None, payment_date_col="pay_date",
                                          currency=ieq.currency, daycounter_convention=tq.DayCounterConvention.Actual365)
iview = tq.Equity(INDEX, ieq.currency, tf.Variable(float(iq["ref_spot"].iloc[0]), dtype=tf.float64),
                  ieq.discount_curve, None, idx_div, None, None)       # the index with the Bloomberg projections only
used = ev.implied_index_dividends(iview, iq)
print("keep years:", ev.DIVIDEND_KEEP_YEARS, "| cutoff", used.attrs["cutoff"], f"| repo held {used.attrs['repo']:.4%}",
      f"| extrapolated yield {used.attrs.get('yield', float('nan')):.4%}")
stored = frames["eq_dividends.csv"].query("ticker == @INDEX")
print("same as eq_dividends.csv:", len(stored) == len(used) and np.allclose(stored["amount"], used["amount"]))
pd.DataFrame(used.attrs["implied"]).round(3)

**Reading the result.**
- **Cutoff and repo:** the cutoff is 2028-11-13, and the repo is held at −0.82%.
- **The table:** for each quoted expiry beyond the cutoff, its forward, the PV of the dividends up to it, and the
  dividends added in the period. With a bid / ask market only December 2028 is beyond the cutoff (the 2029-2032
  expiries have settlement prices only, dropped by §2.4), and it adds nothing: the projections up to the cutoff
  already explain its forward.
- **After December 2028:** the dividends are projected at a constant yield of 3.10% with the repo held, about
  202-210 index points a year (Bloomberg projects about 200 for 2028-2029). Index dividend futures, when added,
  will give these years from the market.

The dividends are identical to those of the snapshot.

The bar chart shows the dividends per year by source:
- the Bloomberg projection;
- the part kept (to 2 years);
- the part implied by the options;
- the extrapolation.

In [ ]:
def by_year(frame, label):
    """Sum of the dividend amounts per calendar year of the ex-date."""
    return frame.assign(year=pd.to_datetime(frame["ex_date"]).dt.year).groupby("year")["amount"].sum().rename(label)


years = pd.concat([by_year(download, "download")] +
                  [by_year(used[used["source"] == s], s) for s in ("download", "implied", "extrapolated")], axis=1)
years.columns = ["Bloomberg projection", "kept", "implied", "extrapolated"]
years.fillna(0).plot.bar(figsize=(10, 3.5), title=f"{INDEX} dividends per year (index points)", width=0.8)
plt.show()

The effect on the repo: the repo implied by the options (§3) on every expiry from 30 days, once with the Bloomberg
projections, once with the dividends above. With the projections alone, the repo falls with maturity, because it is
absorbing missing dividends. With the implied dividends it is flat at $q_0$ beyond the cutoff, by construction.

In [ ]:
used_curve = tq.DividendCurve(mkt.evaluation_date, list(used["ex_date"]), [float(a) for a in used["amount"]],
                              ieq.currency, payment_dates=list(used["pay_date"]),
                              daycounter_convention=tq.DayCounterConvention.Actual365)
ip = ev._prepare_quotes(iq)
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, curve in (("Bloomberg projections", idx_div), ("implied dividends, repo held", used_curve)):
    v = tq.Equity(INDEX, ieq.currency, iview.spot, ieq.discount_curve, None, curve, None, None)
    pts = []
    for e, g in ip.groupby("expiry"):
        m = g[~g["settlement"]]
        if (e - QUOTE_DATE).days >= 30:
            try:
                pts.append(((e - QUOTE_DATE).days / 365, 100 * tq.implied_forward(v, e, m if len(m) else g)["repo"]))
            except ValueError:
                pass
    ax.plot(*zip(*pts), "o-", label=label)
ax.set_xlabel("years"); ax.set_ylabel("implied repo %"); ax.set_title(f"{INDEX}: repo left to the options")
ax.legend(); plt.show()

## 5. The smile model: a mixture of lognormals with a default mass
For each expiry we need the **distribution of the price $S_T$ at the expiry**: it gives the price of every option
of that expiry, hence a vol at every strike (the smile). A curve drawn directly through the market vols could imply
negative probabilities, i.e. arbitrage; a distribution never does. The distribution used is a mixture of
lognormals (Brigo-Mercurio) with a mass at zero.

The section goes in four steps, all on the AAPL December 2027 expiry:
1. the model: what the distribution is (§5.1);
2. the price of one option in the model (§5.2);
3. from that price to a vol, and why the vols form a smile (§5.3);
4. where the parameters come from (§5.4).

### 5.1 The model: scenarios
Think of $S_T$ as drawn in **two stages**:
1. **First a scenario is drawn**: scenario $l$ ($l = 1..4$) with probability $p_l$, or default with probability $Q$.
2. **Then, inside the scenario, the price**:
   - in scenario $l$: lognormal around the scenario's own forward $F_l$, with the scenario's own vol $\sigma_l$, i.e.
     exactly the Black-Scholes world with forward $F_l$ and vol $\sigma_l$;
   - in default: $S_T = 0$.

In formulas:
$$S_T = \begin{cases} 0 & \text{with probability } Q,\\ F_l\,e^{\sigma_l\sqrt T Z - \sigma_l^2 T/2} & \text{with probability } p_l\ (l = 1..4),\end{cases} \qquad Z \sim N(0,1).$$

**The parameters of one expiry:**
- $p_l$: the probability of scenario $l$;
- $F_l$: its forward, the average of $S_T$ in the scenario ($\mathbb E[S_T \mid l] = F_l$);
- $\sigma_l$: its volatility, a yearly vol like a Black vol;
- $Q$: the probability of default.

**Two constraints:**
- $Q + \sum_l p_l = 1$: the probabilities add up to one;
- $\sum_l p_l F_l = F$: the average of $S_T$ over all the scenarios is the market forward of §3 (in default
  $S_T = 0$ adds nothing).

**Relative forwards.** The parameters are stored relative to the forward, $\xi_l = F_l/F$ (so $\sum_l p_l\,\xi_l = 1$):
the `means` of `LognormalMixture` and of `eq_vol_calibration.csv` are these ratios, and its `stdevs` are
$\sigma_l\sqrt T$. In this form the smile of an expiry can be read at another forward: the surface reads each smile
at the forward moneyness $K/F(t)$ (§7), and the extrapolation scales $\ln\xi_l$.

The cell rebuilds the mixture stored by the calibration for this expiry and checks the two constraints.

In [ ]:
calib = frames["eq_vol_calibration.csv"].query("ticker == @TICKER")
row = calib[calib["expiry"] == str(EXPIRY)].iloc[0]                       # stored parameters of the 1Y smile
mix = tq.LognormalMixture(*(np.array(eval(row[c])) for c in ("weights", "means", "stdevs")), row["default"])
p_l, F_l, sigma_l, Q = mix.weights, mix.means * F, mix.stdevs / np.sqrt(t), mix.default
print(pd.DataFrame({"p_l": p_l, "xi_l (stored)": mix.means, "F_l = xi_l F": F_l,
                    "stdev stored = sigma_l sqrt(T)": mix.stdevs, "sigma_l": sigma_l}).round(4))
print(f"Q = {Q:.4%} | Q + sum p_l = {Q + p_l.sum():.12f} | sum p_l F_l = {(p_l * F_l).sum():.6f} (F = {F:.6f})")

**Reading the result.** Four scenarios for AAPL in December 2027 (market forward $F = 376.6$):

| scenario | probability $p_l$ | forward $F_l$ | $\xi_l = F_l/F$ | vol $\sigma_l$ | reading |
|---|---|---|---|---|---|
| crash | 1.9% | 114 | 0.30 | 11% | the stock at about a third, a crash |
| lower | 52% | 349 | 0.93 | 30% | a little lower, with a lot of uncertainty |
| higher | 44% | 416 | 1.11 | 18% | higher, with less uncertainty |
| far up | 2.0% | 461 | 1.22 | 6.7% | well higher, narrow: its vol is on the floor of the fit (§5.4) |
| default | 0.01% | | | | the stock worth zero |

Both constraints hold. The scenarios are not a forecast ("2% chance of a crash to a third"): they are a flexible way
to shape the distribution, and different sets of scenarios can give almost the same smile. What matters is the
overall distribution: before the floor on the vols of §5.4, the fit of the same quotes had a scenario at 18% with
forward 278, 0.5% of default and a far-up scenario of vol 0.5% (a spike of the density at 1.26 F), for a smile
within 0.01 vol points of this one.

The plot shows it:
- **left:** the density of $S_T$ (black), the sum of the four lognormal densities weighted by their probabilities
  (dashed), plus the mass $Q$ at zero. The small bump at 0.3 F is the crash scenario: it lies below the lowest
  quote (49% of the forward), where no quote constrains the model, so its exact shape there is arbitrary;
- **right:** the smile it gives (computed in §5.2-5.3).

In [ ]:
s = np.linspace(0.01, 2.0, 600) * F
dens = sum(p * norm.pdf(np.log(s / (x * F)) + sg**2 / 2, scale=sg) / s for p, x, sg in zip(mix.weights, mix.means, mix.stdevs))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for p, x, sg in zip(mix.weights, mix.means, mix.stdevs):
    ax[0].plot(s / F, p * norm.pdf(np.log(s / (x * F)) + sg**2 / 2, scale=sg) / s * F, lw=0.8, ls="--")
ax[0].plot(s / F, dens * F, "k", label=f"density (+ mass {mix.default:.2%} at 0)")
ax[0].set_xlabel("S_T / F"); ax[0].legend(fontsize=8); ax[0].set_title(f"{TICKER} {EXPIRY}: risk-neutral density")
k = F * np.linspace(0.4, 1.8, 200)
ax[1].plot(k / F, mix.implied_vols(F, k, t), "k")
ax[1].set_xlabel("K / F"); ax[1].set_ylabel("Black vol"); ax[1].set_title("its smile"); plt.tight_layout(); plt.show()

### 5.2 The price of an option: an average of Black prices
An option's price (undiscounted) is the expected value of its payoff. With scenarios, an expected value is the
average over the scenarios of the expected value in each one, weighted by their probabilities. In each scenario
$S_T$ is lognormal with forward $F_l$ and vol $\sigma_l$, so the expected payoff there is just a Black price. So:
1. **price the option in each scenario as if it were the only one**: Black with the scenario's forward $F_l$ and vol
   $\sigma_l$, same strike and expiry. In default ($S_T = 0$) a put pays the whole strike $K$, a call nothing;
2. **average these prices with the probabilities** $p_l$ and $Q$:
$$C(K) = \sum_l p_l\,\mathrm{Black}(F_l, K, \sigma_l, T), \qquad P(K) = \sum_l p_l\,\mathrm{Black}_{\text{put}}(F_l, K, \sigma_l, T) + Q\,K,$$
with $\mathrm{Black}(F, K, \sigma, T)$ the undiscounted Black call price.

The example: the put with strike at 80% of the forward. The cell does the two steps by hand, then compares the result
with the library (`LognormalMixture.price`).

In [ ]:
def black(f, k, vol, t, cp):
    """Undiscounted Black price."""
    sd = vol * np.sqrt(t)
    d1 = np.log(f / k) / sd + sd / 2
    call = f * norm.cdf(d1) - k * norm.cdf(d1 - sd)
    return call if cp == "C" else call - f + k


SCENARIOS = ["crash", "lower", "higher", "far up"]         # the four lognormals of §5.1, in their stored order


def by_scenario(k, cp):
    """Step 1 (price in each scenario) and step 2 (weighted average) for one option."""
    prices = [black(f, k, v, t, cp) for f, v in zip(F_l, sigma_l)] + [k if cp == "P" else 0.0]   # + default
    probs = list(p_l) + [Q]
    return pd.DataFrame({"scenario": SCENARIOS + ["default"],
                         "probability": probs, "forward F_l": list(F_l) + [0.0], "vol sigma_l": list(sigma_l) + [np.nan],
                         "price in the scenario": prices, "probability x price": np.multiply(probs, prices)})


K80 = 0.8 * F
table80 = by_scenario(K80, "P")
price80 = table80["probability x price"].sum()
print(f"put, strike {K80:.2f} (80% of F = {F:.2f})")
display(table80.round(4))
print(f"price = sum of the last column = {price80:.4f} | library LognormalMixture.price: "
      f"{float(mix.price(F, [K80], 1.0, 'P')[0]):.4f}")

**Reading the result.** The put with strike 301.3:
- **In the crash scenario** (forward 114) the put is deep in the money: it is worth 186.87, about the strike minus
  the forward.
- **In the lower scenario** (forward 349, vol 30%) it is out of the money but within reach: 20.72.
- **In the higher scenario** (forward 416, vol 18%) it is far out of the money: 1.00.
- **In the far up scenario** it is worth nothing.
- **In default** it would pay the whole strike, 301.3, but default has a probability of 0.01%.

The average, with the probabilities, gives the price:
$0.019 \times 186.87 + 0.519 \times 20.72 + 0.443 \times 1.00 + 0.020 \times 0 + 0.0001 \times 301.3 = 3.48 + 10.75 +
0.44 + 0 + 0.04 = 14.71$. The library gives the same.

Three quarters of the price come from the lower scenario, a quarter from the crash.

### 5.3 From the price to a vol: the smile
The market quotes vols, not prices. So, for each strike, the model price is turned into a vol: the **single** Black
vol that, with the market forward $F$ (one forward, not the scenario forwards), gives the same price. That is the
model's implied vol at that strike, the number the fit (§6) compares with the market vol.

If the distribution were one lognormal, this vol would be the same at every strike. The mixture is not one
lognormal, so the vol changes with the strike, and that is the smile. Why, on the 80% put:
- priced with one lognormal at the at-the-money vol of the mixture, the put would be worth less;
- in the mixture it is worth more, because the down scenario and default make a price below 301 much more likely;
- to give that higher price with one lognormal, a higher vol is needed.

The cell does this for the 80% put, then for five strikes, with the share of each scenario in the price.

In [ ]:
atm_vol = float(mix.implied_vols(F, [F], t)[0])
iv80 = float(tq.black_implied_vol(np.array([price80]), F, np.array([K80]), t, "P")[0])
print(f"80% put: mixture price {price80:.2f} -> implied vol {iv80:.2%}")
print(f"         one lognormal at the ATM vol {atm_vol:.2%} would give {black(F, K80, atm_vol, t, 'P'):.2f}")

rows = []
for m, cp in ((0.6, "P"), (0.8, "P"), (1.0, "C"), (1.2, "C"), (1.4, "C")):        # out-of-the-money options
    tab = by_scenario(m * F, cp)
    price = tab["probability x price"].sum()
    share = 100 * tab["probability x price"] / price
    rows.append({"option": f"{'put' if cp == 'P' else 'call'} {m:.0%} F", "price": price,
                 **{f"% {n}": v for n, v in zip(tab["scenario"], share)},
                 "implied vol": float(tq.black_implied_vol(np.array([price]), F, np.array([m * F]), t, cp)[0]),
                 "library vol": float(mix.implied_vols(F, [m * F], t)[0])})
pd.DataFrame(rows).round(4)

**Reading the result.**
- **The 80% put.** The mixture price, 14.71, corresponds to a vol of 30.0%. One lognormal at the ATM vol (26.8%)
  would price it at 11.34: the mixture puts more probability below 301, so the put is dearer, hence a higher vol.
- **The five strikes.** Each column `% ...` is the share of a scenario in the price:
  - **low-strike puts (60%, 80%):** the price comes from the crash scenario (55% at 60%, 24% at 80%) and the
    high-vol lower scenario: a fat left tail, so high vols: 34.7% and 30.0%;
  - **at the money:** the lower and higher scenarios (40% and 56%): 26.8%;
  - **high-strike calls (120%, 140%):** the higher scenario has a low vol (18%) and the far up one is narrow: less
    probability far above the forward than one lognormal would give, so lower vols: 24.8% and 24.1%.
- The implied vols equal the library's (`library vol`): this is how `LognormalMixture.implied_vols` works.

The vol falling from the low strikes to the high ones is the skew of the smile.

Last, a check of the limit case: one lognormal and no default is plain Black, i.e. a flat smile.

In [ ]:
one = tq.LognormalMixture([1.0], [1.0], [0.3 * np.sqrt(t)], 0.0)
print("one lognormal at 30%, no default -> vols:", one.implied_vols(F, F * np.linspace(0.5, 1.6, 9), t).round(6))

### 5.4 Where the parameters come from
Nobody chooses $p_l$, $F_l$, $\sigma_l$ and $Q$: they are the free parameters of the model, found by the fit (§6) so
that the model reproduces the market vols of the expiry:
1. from the market: the out-of-the-money options of the expiry, each with its implied vol (20 for this expiry);
2. start from initial parameters (four ordered scenarios, $Q \approx 0.1\%$);
3. with those parameters, the model vol at each of the 20 strikes (§5.2-5.3);
4. compare with the market vols;
5. the optimizer (least squares) changes the parameters until the sum of the squared errors is the smallest.

The constraints always hold by construction:
- the probabilities are positive and add up to one;
- the forwards are rescaled so that $\sum_l p_l F_l = F$;
- the vols stay between a quarter of the at-the-money vol (and at least 0.5%) and 300%.

**Why the floor on the vols.** The fit has more freedom (12 free parameters) than the quotes need where the strikes
are sparse. Without a floor it would put a scenario on a vol near zero (0.5%), an almost point mass in the density,
for a gain of hundredths of a vol point: on this expiry, 1% of probability on 475 = 1.26 F (a spike in the density),
and up to 12% on other expiries of the snapshot. With the floor at a quarter of the at-the-money vol (6.7% here) the
density is smooth, the fit error grows by hundredths of a vol point at most, and the quotes repriced within bid / ask
are the same. The far up scenario of §5.1 sits on that floor.

The parameters of §5.1 are where the fit stopped for this expiry: there the model vols match the market ones within
0.03 vol points. Another day, or another expiry, gives other parameters.

## 6. Fit of the smile
How the mixture of §5 is fitted to the quotes of one expiry, inside `build_equity_vol`:
1. **Out-of-the-money quotes** relative to the implied forward $F$ of §3: puts below $F$, calls above. They are
   the liquid ones, and an in-the-money option adds only the forward to an out-of-the-money one.
2. **Implied vols.** Each mid (and bid and ask) becomes a Black vol on $F$. For an American option, it is the vol
   of its American price on $F$. That is also the vol of the European option, so it is the Black vol.
3. **Outliers.** Vols that cannot be computed, or outside $0.2$–$5\times$ the median vol of the expiry, are
   dropped. An expiry with fewer than 3 quotes left is skipped (the surface interpolates through it).
4. **Least squares**, with weights $w_i$ (1 for a two-sided quote; 0.25 for a settlement price, used only without
   a bid / ask market, §2.4), on
   $$\sum_i \Big(w_i\,\frac{V_{\text{model}}(K_i) - \text{Black}(F, K_i, \sigma_i)}{\mathcal V_i}\Big)^2.$$
   The price errors divided by the vega $\mathcal V_i$ are about the vol errors, but each step needs only Black
   prices (no implied-vol inversion inside the optimizer).

The fit uses the mids; the bid / ask vols are only shown against the result.

### 6.1 One expiry by hand
The 1Y expiry again: the four steps above with the public functions (`tq.implied_volatility`,
`tq.LognormalMixture.fit`), then the comparison with the stored fit.

In [ ]:
otm = grp[np.where(grp["option_type"] == "P", grp["strike"] < F, grp["strike"] >= F)].sort_values("strike")   # 1.
vols = {}
for name, price in (("bid", otm["bid"]), ("mid", 0.5 * (otm["bid"] + otm["ask"])), ("ask", otm["ask"])):        # 2.
    vols[name] = np.concatenate([tq.implied_volatility(price[otm["option_type"] == cp].to_numpy(), otm.loc[otm["option_type"] == cp, "strike"].to_numpy(),
                                                       cp, REF_SPOT, r, q_carry, t, american=True) for cp in ("P", "C")])
strikes = np.concatenate([otm.loc[otm["option_type"] == cp, "strike"].to_numpy() for cp in ("P", "C")])
settle = np.concatenate([otm.loc[otm["option_type"] == cp, "settlement"].to_numpy() for cp in ("P", "C")])
v = vols["mid"]
median = np.nanmedian(v)
keep = np.isfinite(v) & (v > 0.2 * median) & (v < 5 * median)                                                  # 3.
w = np.where(settle, 0.25, 1.0)
fit = tq.LognormalMixture.fit(F, t, strikes[keep], v[keep], n=4, weights=w[keep])                             # 4.
print(f"{keep.sum()} of {len(v)} quotes used (median vol {median:.2%}), {settle.sum()} settlements | rms {fit.rms:.4%}"
      f" (stored {row['fit_rms']:.4%})")
table = pd.DataFrame({"K/F": strikes / F, "settlement": settle, "used": keep, "weight": w,
                      **{f"iv_{n}": vols[n] for n in vols}, "model": fit.implied_vols(F, strikes, t)})
table.assign(**{"error bp": 1e4 * (table["model"] - table["iv_mid"])}).round(4)

**Reading the result.**
- **Quotes:** 20 out-of-the-money quotes from 49% to 149% of the forward, none dropped.
- **Fit error:** 0.031% rms, i.e. 0.03 vol points, the same as the stored fit. The largest error is about 7 bp of
  vol (`error bp`), while the half bid / ask is about 25-120 bp of vol: the smile goes through the quotes, well
  inside their spreads.
- **Shape:** the skew goes from 38.6% at 49% to 24.1% at 149%.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
kk = F * np.linspace(strikes.min() / F * 0.95, strikes.max() / F * 1.05, 200)
ax.vlines(strikes / F, vols["bid"], vols["ask"], color="grey", lw=2, alpha=0.5, label="bid / ask")
ax.scatter((strikes / F)[keep & ~settle], v[keep & ~settle], s=18, label="mid (w = 1)")
ax.scatter((strikes / F)[keep & settle], v[keep & settle], s=22, marker="s", label="settlement (w = 0.25)")
ax.scatter((strikes / F)[~keep], np.nan_to_num(v[~keep]), marker="x", color="r", label="dropped")
ax.plot(kk / F, fit.implied_vols(F, kk, t), "k", label=f"mixture, rms {fit.rms:.2%}")
ax.set_xlabel("K / F"); ax.set_ylabel("vol"); ax.set_title(f"{TICKER} {EXPIRY}"); ax.legend(fontsize=8); plt.show()

### 6.2 Every expiry (`build_equity_vol`)
`tq.build_equity_vol(view, quotes)` runs §2.4, §3 and §6 on every expiry, then §7. A least-squares fit can stop
in a local minimum, so two safeguards are used:
- **The start.** It is ordered: from the lowest forward $F_l$ with the highest vol $\sigma_l$ to the highest forward
  with the lowest. That removes the symmetry of exchanging two scenarios.
- **The warm start.** When the fit is above 0.2 vol points rms, a second fit starts from the parameters of the
  previous expiry, and the better of the two is kept.

The report has one row per expiry:
- its forward and repo;
- the number of quotes used, and their source (`market` or `settlement`);
- the rms of the fit;
- the ATM vol;
- the default probability $Q$.

In [ ]:
surface, repo_curve, report = tq.build_equity_vol(view, quotes)
good = report[report["mixture"].notna()]                                   # calibrated expiries
report[["expiry", "t", "forward", "repo", "quotes", "source", "fit_rms", "atm_vol"]].assign(
    default=[m.default if isinstance(m, tq.LognormalMixture) else np.nan for m in report["mixture"]]).round(4)

**Reading the result.**
- **Rms:** about 0.1 vol points at most (0.10% in December 2028), so the warm start is never needed here.
- **$Q$:** mostly grows with maturity, from about 0.01% at a few days to 2.1% in December 2028: a default
  probability accumulates with time. It is not identified on its own, though: the far left tail is beyond the
  quotes, and the fit can put that probability at zero (default) or on a crash scenario (as in December 2027,
  0.01%). On names with a steep short-dated put skew it is the mass at zero that bends the far puts of the first
  expiries.
- **Repo:** above 1% on the two weeklies (3 and 5 days), noise of a few cents on the forward. That is why they give
  no repo pillar (§3.2).

The panels below show every expiry: the quotes (mid points with their bid / ask bars) and the fitted smile.

In [ ]:
n = len(good)
fig, axes = plt.subplots(int(np.ceil(n / 4)), 4, figsize=(14, 2.6 * np.ceil(n / 4)), sharey=False)
for ax, (_, g) in zip(axes.ravel(), good.iterrows()):
    qv = g["quote_vols"]                                                   # per quote: bid / mid / ask vols, used
    x = qv["strike"] / g["forward"]
    ax.vlines(x, qv["iv_bid"], qv["iv_ask"], color="grey", alpha=0.5)
    ax.scatter(x[qv["used"]], qv.loc[qv["used"], "iv_mid"], s=8)
    kk = g["forward"] * np.linspace(x.min(), x.max(), 100)
    ax.plot(kk / g["forward"], g["mixture"].implied_vols(g["forward"], kk, g["t"]), "k", lw=1)
    ax.set_title(f"{g['expiry']}  rms {g['fit_rms']:.2%}", fontsize=9)
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. The surface
The pricers read the vols on a fixed grid (bilinear between the nodes, flat outside), so the smiles of §6 are
carried onto it.

**The grid.**
- **Maturities:** the calibrated expiries, then one point a year up to 6 years.
- **Strikes:** 30%–200% of the spot, every 5% below 95% (the autocallable barriers sit at 50-70%), then 97.5%,
  100%, ..., 150%, 175% and 200%: 26 strikes.

Each node is read at its **forward moneyness** $K/F(t)$, with the forward of the repo curve of §3.

**Each smile beyond its quotes.** Between the lowest and the highest quote of an expiry the smile is the mixture.
Beyond them no quote constrains the mixture, and its tail can do anything (bend down, carry a bump). So there the
total variance $w = \sigma^2 t$ continues as a **straight line in $k = \ln(K/F)$**, from the edge of the quotes with
the slope of the mixture there:
- the slope is kept within Lee's bounds ($|w'| \le 2$) and never makes $w$ decrease away from the money;
- a straight line drops the curvature of the smile at the edge, and that can make the density negative (Gatheral's
  $g(k) = (1 - k w'/2w)^2 - w'^2/4\,(1/w + 1/4) + w''/2 < 0$). Where it would, the line starts further out (the
  mixture kept up to that point, a density near the quotes);
- if no starting point works, the mixture's own tail is kept (a density, so free of arbitrage) and marked
  **unreliable**. It happens on expiries of a few days quoted only near the money.

**How each node is filled, by maturity:**
- **Calendar step.** At the same forward moneyness an expiry must have at least the total variance of the earlier
  ones (calendar arbitrage). Where it has less it is raised to them; only raised, because lowering part of a smile
  would make its density negative (butterfly arbitrage). A reliable point (quoted, or on a straight wing) is
  raised only by earlier reliable points: an unreliable tail never moves a quote nor a reliable wing.
- **Between two expiries.** Total variance $\sigma^2 t$, interpolated in time by a monotone cubic (PCHIP) through
  the smiles of all the expiries, which keeps the monotonicity of the calendar step.
- **Before the first expiry.** The first smile.
- **After the last expiry $T_n$.** The last mixture is carried forward: each scenario keeps its vol $\sigma_l$, and
  $\ln\xi_l = \ln(F_l/F)$ grows with $\sqrt{t/T_n}$.

**The floor past $T_n$.** A short last expiry would otherwise freeze its level forever. So the ATM vol is
compared with a Heston-like term structure fitted to the ATM vols of all the expiries,
$$\sigma_H^2(t) = \theta + (v_0 - \theta)\,\frac{1 - e^{-\kappa t}}{\kappa t},$$
where the variance starts at $v_0$ and reverts to $\theta$ at speed $\kappa$. If the carried ATM vol is below
$\sigma_H(t)$, all the scenario vols are scaled up to reach it.

**Expiries with a wing free of quotes.** A very short expiry quoted only near the money (none outside 90%-110% of
the forward), or a long one with puts only, has nothing that shapes a wing. The smile of the next complete expiry
(the previous one for the last expiries) is carried to its maturity (vols of the scenarios kept, $\ln\xi_l$ scaled by
$\sqrt{t/T}$; for a short one scaled to its own at-the-money vol) and guides the fit: points of that smile beyond the
quotes, with a low weight (0.1), and the start of the fit. The expiry's own quotes are fitted as usual. The report
shows the lender in `smile_from`.

**After the last expiry,** the edges of the quotes are carried in standardised moneyness $\ln(K/F)/\sqrt t$ and the
same wings apply; the result is never below the last expiry.

**Extreme strikes.** Where a vol cannot be inverted, it is linear in log-moneyness from the two nearest nodes.

The cell shows the AAPL grid and checks it against `eq_vol.csv`.

In [ ]:
stored_vol = frames["eq_vol.csv"].query("ticker == @TICKER")
matrix = surface.volatility_matrix.numpy()                                  # [maturities, strikes]
print("grid:", len(surface.maturity), "tenors x", len(surface.strike), "strikes | moneyness", np.round(ev.GRID_MONEYNESS, 2))
print("max |surface - eq_vol.csv| =", np.abs(np.sort(matrix.ravel()) - np.sort(stored_vol["vol"].to_numpy())).max())
pd.DataFrame(matrix, index=np.round(surface.maturity, 3), columns=np.round(np.array(surface.strike) / REF_SPOT, 2)).round(4)

**Reading the result.** The grid is 16 maturities × 26 strikes:
- **Maturities:** the 12 expiries, then 3, 4, 5 and 6 years.
- **Check:** equal to the stored surface to $10^{-15}$.

Along the rows:
- **First rows (3-7 days):** high vols in the wings, up to 186% at 30% on the 7-day row: on a few days a 30% strike
  is very far in standard deviations, so a tiny price is a huge vol. The 3- and 5-day expiries are quoted only
  between 95% and 105% of the forward: their wings are guided by the smile of 20 November (see below), not left to
  a free mixture tail (that gave up to 384%).
- **From 1 month:** the wings are the straight lines beyond the quotes: at one year the vol rises from 39.2% at 50%
  to 48.4% at 30%, steadily (no more bending down below 40%, as the mixture's own tail did).
- **From 1 year:** the smile flattens.
- **From 3 years:** the rows move only slowly. That is the extrapolation, analysed below.

The ATM term structure for AAPL and SX5E, side by side:
- **Dots:** the ATM vol of each calibrated expiry.
- **Line:** $\sigma_H(t)$ fitted to them.
- **Dashed line:** the last mixture carried past $T_n$ without the floor.
- **Black points:** the surface at the forward.

AAPL stops at about 2.2 years, so its long end is the extrapolation. SX5E is listed to 6 years and needs none.

In [ ]:
def atm_term(ticker, good, surface, view, ax):
    """ATM vols of the calibrated expiries, sigma_H(t), the carried last mixture and the surface at the forward."""
    times, atm = good["t"].to_numpy(float), good["atm_vol"].to_numpy(float)
    sigma_h = ev._heston_atm(times, atm)                              # (kappa, v0, theta) fitted to the ATM vols
    tt = np.linspace(times[0], 6.2, 300)
    after = tt[tt > times[-1]]
    last = good["mixture"].iloc[-1]
    carried = [float(ev._extrapolate(last, times[-1], x).implied_vols(1.0, [1.0], x)[0]) for x in after]
    grid_t = np.array(surface.maturity)
    atm_grid = [float(surface.volatility(strike=float(view.forward(x)), tenor=x)) for x in grid_t]
    ax.plot(times, atm, "o", label="ATM vol of each calibrated expiry")
    ax.plot(tt, [sigma_h(x) for x in tt], label="Heston-like sigma_H(t)")
    if len(after):
        ax.plot(after, carried, "--", label="last mixture carried (no floor)")
        ax.axvline(times[-1], color="grey", lw=0.6)
    ax.plot(grid_t, atm_grid, "k.", label="surface at the forward")
    ax.set_xlabel("years"); ax.legend(fontsize=8); ax.set_title(f"{ticker}: ATM term structure")


idx_quotes, idx_view = equity_inputs(INDEX)                      # dividends of the market: the implied ones (§4)
idx_surface, _, idx_report = tq.build_equity_vol(idx_view, idx_quotes)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
atm_term(TICKER, good, surface, view, ax[0])
atm_term(INDEX, idx_report[idx_report["mixture"].notna()], idx_surface, idx_view, ax[1])
plt.tight_layout(); plt.show()

**Reading the result.**
- **AAPL, past the grey line (the last expiry, about 2.2 years).** The carried mixture drifts down, from 26.7% to
  26.5% at 6 years. $\sigma_H$ rises slowly towards its long-run level, from 27.1% to 27.4%. The carried vol is below
  the floor everywhere, so the surface is raised to $\sigma_H$ (about 27.3%-27.5% at the forward from 3 years).
  The floor only raises vols, never lowers them.
- **SX5E.** Listed to 6.1 years: the whole grid lies inside the listed expiries, so nothing is extrapolated, and
  $\sigma_H$ is just a smooth line through the ATM vols.

Last, the AAPL smiles on the grid (one curve every few maturities): steep and curved at the short end, flatter
with maturity. That is the usual equity pattern: skew in absolute terms decreasing roughly as $1/\sqrt T$.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
grid_t = np.array(surface.maturity)
for i in range(0, len(grid_t), max(1, len(grid_t) // 8)):
    ax.plot(ev.GRID_MONEYNESS, matrix[i], label=f"{grid_t[i]:.2f}y")
ax.set_xlabel("K / S0"); ax.set_ylabel("vol"); ax.set_title(f"{TICKER}: smiles on the grid"); ax.legend(fontsize=8, ncol=2)
plt.show()

## Exercises
1. Build the SX5E surface keeping its settlement-only expiries 2029-2032 (set `bid` = `ask` = `last` on them before
   `build_equity_vol`): how do the long smiles, the repo and the implied dividends compare with the projections?
2. Recompute the AAPL forward and repo with `dividend_model="discrete"` (`tq.implied_forward(..., dividend_model=...)`):
   does the forward change? Does the American implied vol?
3. Take SX5E with the Bloomberg projections only (no implied dividends, `iview` of §4): how does the long-dated repo
   look, and what would it do to the 5Y forward?